# ODOT SCD Components — Functional Verification

This notebook exercises every ODOT Standard Construction Drawing (SCD)
component civilpy has built (`src/civilpy/structural/odot/*.py`): for each
one, it calls the catalog lookup(s) and the pure-Python `layout_*()`
generator with representative inputs, and checks the result against the
transcribed drawing data (spacing, counts, formulas, guarded lookups).

This is **not** a replacement for `pytest tests/structural` (which is the
authoritative, CI-run test suite — see `docs/SCD_BUILD_LOG.md` for the
per-SCD test counts) — it's a single narrative pass over the whole SCD
program so a person (or a future session) can see every component's
inputs/outputs side by side, the way the "Rhino → MIDAS Pipeline
Verification" notebook in this same folder does for the girder pipeline.

Everything here runs **offline** — no Rhino needed, since all engineering
content lives in pure-Python `civilpy.structural.odot.*` modules. The
Grasshopper (`Notebooks/res/*.py`) scripts are thin drawing layers over
these same functions and are not re-tested here (they need a live Rhino
8 session with `rhino3dm`/`Rhino.Geometry` available).

In [11]:
def check(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, name

def approx(a, b, tol=1e-6):
    """True if a and b agree within a relative/absolute tolerance -- a
    dependency-free stand-in for pytest.approx for this narrative notebook."""
    return abs(a - b) <= tol * max(1.0, abs(b))

report = {}

""" General TODOs
# //TODO - Most of the imported civilpy libraries based on these SCDs or AASHTO 
specs need to take in "des_yr" as an optional input, many of the AASHTO Checks 
already do. They're all being designed with the most modern SCD we currently 
have `des_yr=2026` but a long term goal will be to be able to support quickly 
generating structures that have been built previously/are exisiting.
""";

## A-1-20 — Typical Abutment Detail (Expansion Joints)

`odot.typical_abutment`: **guidance only** (the sheet's own note says not
to use it as a standalone construction drawing) — the bearing-seat and
wingwall-limit formulas, section minimums, for a visual check only.

In [6]:
from civilpy.structural.odot.typical_abutment import (
    AbutmentInput as TypicalAbutmentInput, bearing_seat_dim_a_ft, layout_typical_abutment,
)

"""
# //TODO - Should mostly be handled by the TODOs in Notebooks/res/A-1-20.py but
Missing a lot of inputs on these functions, Near/Far Abutment determines which 
way the wingwalls would be swept, probably needs to take in an alignment and a 
terrain model to be able to understand what kind of slope is required on the 
wingwall itself. That's a large task to build. Terrain models can be automatically
generated from OGRIP Data, I think CivilPy Already has existing infrastructure to
handle that. The rebar isn't coming through in the Rhino model but appears to exist
in the python object when it's inspected. It should probably be independent of any 
approach slab values, I'm not sure why it's depicted as having it inherited by composition.
Not able to tell if any (and which if any are) design checks are being implemented when
this function is being utilized, probably one of the most difficult standards to successfully
implement.
"""

check("A-1-20: DIM.A grows with skew", bearing_seat_dim_a_ft(30.0) > bearing_seat_dim_a_ft(0.0))

layout = layout_typical_abutment(TypicalAbutmentInput(
    width_ft=30.0, skew_deg=15.0, wingwall_length_ft=6.0, footing_depth_ft=3.0,
    backwall_height_ft=5.0,
))
check("A-1-20: wingwall springs from backwall end", layout.wingwall_outline[0] == layout.backwall_outline[2])
report["A-1-20"] = 1

  [PASS] A-1-20: DIM.A grows with skew
  [PASS] A-1-20: wingwall springs from backwall end


## AS-1-15 — Reinforced Concrete Approach Slab

`odot.approach_slab`: the reinforcing-steel table (spans 15/20/25/30 ft),
bracketed bar-count formulas, seat/joint catalog, and `layout_approach_slab`.

In [4]:
from civilpy.structural.odot.approach_slab import (
    ApproachSlabInput, approach_slab_design, layout_approach_slab,
)

design = approach_slab_design(25.0)
check("AS-1-15: 25 ft design found", design.length_ft == 25.0)

layout = layout_approach_slab(ApproachSlabInput(
    length_ft=25.0, width_ft=40.0, skew_deg=15.0, end_thickness_in=design.thickness_in,
))
check("AS-1-15: profile has no duplicate points", len(layout.profile) == len(set(layout.profile)))
check("AS-1-15: bars generated", len(layout.bars) > 0)
marks = {b.mark for b in layout.bars}
check("AS-1-15: bottom (A) and B501 bar marks present", {"A1003", "B501"}.issubset(marks))
report["AS-1-15"] = len(layout.bars)


  [PASS] AS-1-15: 25 ft design found
  [PASS] AS-1-15: profile has no duplicate points
  [PASS] AS-1-15: bars generated
  [PASS] AS-1-15: bottom (A) and B501 bar marks present


## DS-1-92 — Stainless Steel Drip Strip

`odot.drip_strip`: railing-dependent placement catalog, perforation pattern,
and `drip_strip_runs` (fascia run generator).

In [3]:
from civilpy.structural.odot.drip_strip import (
    PLACEMENTS, drip_strip_runs, placement, strip_profile_in,
)

p = placement("DBR-2-73")
check("DS-1-92: DBR-2-73 placement found", p.railing == "DBR-2-73")

runs = drip_strip_runs(120.0, (10.0, 30.0, 50.0, 70.0, 90.0, 110.0), "DBR-2-73")
check("DS-1-92: runs generated", len(runs) > 0)
prof = strip_profile_in("upper", bent=True)
check("DS-1-92: bent profile has points", len(prof) >= 3)
report["DS-1-92"] = len(runs)


  [PASS] DS-1-92: DBR-2-73 placement found
  [PASS] DS-1-92: runs generated
  [PASS] DS-1-92: bent profile has points


## PCB-91 — Portable Concrete Barrier

`odot.portable_barrier`: the NJ-shape section profile, segment/joint/anchor
layout (crash levels stay in `bridge_railing`).

In [4]:
from civilpy.structural.odot.portable_barrier import (
    anchor_hole_stations_ft, barrier_run, profile_points_in, run_length_ft,
)

prof = profile_points_in(chamfered=True)
check("PCB-91: profile is closed/mirror-symmetric", prof[0][0] == -prof[-1][0])

run = barrier_run(4, segment_length_ft=10.0, joint_gap_in=0.25)
check("PCB-91: 4-segment run built", len(run) == 4)
check("PCB-91: run length matches segments+gaps", approx(run_length_ft(run), 4*10.0 + 3*0.25/12.0))
anchors = anchor_hole_stations_ft(10.0)
check("PCB-91: anchor hole stations found", len(anchors) > 0)
report["PCB-91"] = len(run)


  [PASS] PCB-91: profile is closed/mirror-symmetric
  [PASS] PCB-91: 4-segment run built
  [PASS] PCB-91: run length matches segments+gaps
  [PASS] PCB-91: anchor hole stations found


## AS-2-15 — Approach Slab Installation (Sleeper Slab)

`odot.sleeper_slab`: Type A/C reinforced concrete sleeper slab under the
approach-slab/pavement joint (Type B has none — raises `ValueError`).

In [5]:
from civilpy.structural.odot.sleeper_slab import SleeperSlabInput, layout_sleeper_slab

layout = layout_sleeper_slab(SleeperSlabInput(width_ft=24.0, skew_deg=20.0, installation="A"))
check("AS-2-15: outline is a parallelogram (4 pts)", len(layout.outline) == 4)
check("AS-2-15: SS501/SS502 bars present", {"SS501", "SS502"}.issubset({b.mark for b in layout.bars}))

try:
    layout_sleeper_slab(SleeperSlabInput(width_ft=24.0, installation="B"))
    check("AS-2-15: Type B correctly has no sleeper slab", False)
except ValueError as exc:
    check("AS-2-15: Type B raises (no sleeper slab)", "Type B" in str(exc))
report["AS-2-15"] = len(layout.bars)


  [PASS] AS-2-15: outline is a parallelogram (4 pts)
  [PASS] AS-2-15: SS501/SS502 bars present
  [PASS] AS-2-15: Type B raises (no sleeper slab)


## HW-2.1 / HW-2.2 — Half-Height Headwalls

`odot.headwall`: circular-pipe headwall (end treatment "A") for corrugated
metal/plastic pipe (HW-2.1) and concrete pipe (HW-2.2, `concrete=True`).

In [6]:
from civilpy.structural.odot.headwall import HeadwallInput, layout_headwall

hw21 = layout_headwall(HeadwallInput(diameter_in=36.0, concrete=False))
check("HW-2.1: pipe opening below cover minimum satisfied", hw21.cover_in >= 6.0)

hw22 = layout_headwall(HeadwallInput(diameter_in=36.0, concrete=True))
check("HW-2.2: concrete-pipe table used", hw22.table is not hw21.table or hw22.concrete_cy != hw21.concrete_cy)

try:
    layout_headwall(HeadwallInput(diameter_in=60.0, concrete=False))
    check("HW-2.1: D=60 in should exceed treatment-A cover limit", False)
except ValueError as exc:
    check("HW-2.1: D=60 in correctly raises (end treatment B not modeled)", "treatment" in str(exc).lower())
report["HW-2.1/HW-2.2"] = 2


  [PASS] HW-2.1: pipe opening below cover minimum satisfied
  [PASS] HW-2.2: concrete-pipe table used
  [PASS] HW-2.1: D=60 in correctly raises (end treatment B not modeled)


## HW-1.1 — Full-Height Headwalls

`odot.full_height_headwall`: pipe-diameter x skew-angle dimension/quantity
table (42-84 in, skew 0-45 deg); Type A symmetric / Type B asymmetric
wingwalls, skew bucket pinned to the sheet's own 10 deg Type A/B cutoff.

In [7]:
from civilpy.structural.odot.full_height_headwall import (
    HeadwallInput as FHHInput, layout_full_height_headwall, nearest_skew_bucket,
)

check("HW-1.1: 9 deg skew -> Type A bucket (0)", nearest_skew_bucket(9.0) == 0.0)
check("HW-1.1: 11 deg skew -> Type B bucket (15)", nearest_skew_bucket(11.0) == 15.0)

square = layout_full_height_headwall(FHHInput(diameter_in=60.0, skew_deg=0.0))
check("HW-1.1: Type A is symmetric", approx(square.wing1[2][0], -square.wing2[2][0]))

skewed = layout_full_height_headwall(FHHInput(diameter_in=60.0, skew_deg=30.0))
check("HW-1.1: Type B wingwalls are asymmetric", skewed.type_ == "B")
report["HW-1.1"] = 2


  [PASS] HW-1.1: 9 deg skew -> Type A bucket (0)
  [PASS] HW-1.1: 11 deg skew -> Type B bucket (15)
  [PASS] HW-1.1: Type A is symmetric
  [PASS] HW-1.1: Type B wingwalls are asymmetric


## BCHW — Precast Box Culvert Headwall/Wingwall

`odot.box_culvert_headwall`: a detailing template (every overall dimension
is project-supplied, no catalog) plus the TYPE-1..TYPE-8 rebar bend legend.

In [8]:
from civilpy.structural.odot.box_culvert_headwall import (
    WingwallInput, bend_shape, layout_wingwall,
)

pts = bend_shape("TYPE-8", A=12.0, B=18.0, skew_deg=20.0)
check("BCHW: TYPE-8 corner bar bend generated", len(pts) == 3)

layout = layout_wingwall(WingwallInput(
    length_ft=10.0, skew_deg=15.0, wall_height_ft=8.0, foreslope_height_ft=4.0,
    cutoff_wall_height_ft=2.0, footing_width_ft=6.0, box_wall_thickness_in=12.0,
))
check("BCHW: wingwall footprint generated", len(layout.wingwall_outline) == 4)
report["BCHW"] = len(pts)


  [PASS] BCHW: TYPE-8 corner bar bend generated
  [PASS] BCHW: wingwall footprint generated


## Bridge Railings (SBR/BR/TST/DBR/TBR/PCB) — the shared `build_barriers()` pipeline

`odot.bridge_railing` already cataloged every Wave-3 railing; the generic
`rhino_barrier.shape_family()`/`barrier_profile()` dispatch draws any of
them. This section re-verifies the **BR-2-15 fix**: the "combination"
family (full-height concrete barrier + steel tube rail on top) used to be
misclassified as a bare "steel tube" railing because its shape string
contains the substring "tube".

In [9]:
from civilpy.structural.odot.bridge_railing import BRIDGE_RAILINGS, railing
from civilpy.structural.rhino_barrier import barrier_profile, shape_family

for name in ("BR-1 (36 in)", "SBR-1 (42 in)", "TST-2 (three steel tube)",
             "PCB (portable, unanchored)", "BR-2 (sidewalk barrier + twin tube)"):
    r = railing(name)
    fam = shape_family(r)
    print(f"  {name:38s} -> {fam}")

br2 = railing("BR-2 (sidewalk barrier + twin tube)")
check("BR-2-15: classified as 'combination', not 'steel tube'", shape_family(br2) == "combination")
prof = barrier_profile(br2, 42.0 / 12.0, side=+1)
check("BR-2-15: full 42 in barrier height (not a 10 in curb)",
      approx(max(z for _, z in prof), 3.5))
report["bridge_railing"] = len(BRIDGE_RAILINGS)


  BR-1 (36 in)                           -> new jersey
  SBR-1 (42 in)                          -> single slope
  TST-2 (three steel tube)               -> steel tube
  PCB (portable, unanchored)             -> portable
  BR-2 (sidewalk barrier + twin tube)    -> combination
  [PASS] BR-2-15: classified as 'combination', not 'steel tube'
  [PASS] BR-2-15: full 42 in barrier height (not a 10 in curb)


## SB-1-24 — Single Span Slab Bridges

`odot.slab_bridge`: full SLAB DATA + EDGE BEAM SLAB DATA tables (spans
11-38 ft), skewed parallelogram plan, A/B/M/N longitudinal bar mats.

In [10]:
from civilpy.structural.odot.slab_bridge import SlabBridgeInput, layout_slab_bridge

layout = layout_slab_bridge(SlabBridgeInput(span_ft=24, width_ft=30.0, skew_deg=15.0))
check("SB-1-24: thickness matches the table", layout.thickness_in == 18.25)
check("SB-1-24: all 4 bar marks present", {"A", "B", "M", "N"} == {b.mark for b in layout.bars})
check("SB-1-24: bridge length > span (skew adds length)", layout.bridge_length_ft > 24.0)
report["SB-1-24"] = len(layout.bars)


  [PASS] SB-1-24: thickness matches the table
  [PASS] SB-1-24: all 4 bar marks present
  [PASS] SB-1-24: bridge length > span (skew adds length)


## CS-1-24 — Continuous Slab Bridges

`odot.continuous_slab_bridge`: the largest table in the SCD program
(779 numeric entries, end spans 14-46 ft, interior span fixed at 1.25x
end span); A/B/C/D/E longitudinal bar mats, two piers.

In [11]:
from civilpy.structural.odot.continuous_slab_bridge import (
    ContinuousSlabInput, interior_span_ft, layout_continuous_slab,
)

check("CS-1-24: interior span is 1.25x end span", approx(interior_span_ft(24), 30.0))

layout = layout_continuous_slab(ContinuousSlabInput(end_span_ft=24, width_ft=30.0, skew_deg=10.0))
check("CS-1-24: total length = 2*end + interior",
      approx(layout.total_length_ft, 2*24 + interior_span_ft(24)))
check("CS-1-24: two pier stations", len(layout.pier_stations) == 2)
check("CS-1-24: E-bars present (span 24 >= 22)", "E" in {b.mark for b in layout.bars})
report["CS-1-24"] = len(layout.bars)


  [PASS] CS-1-24: interior span is 1.25x end span
  [PASS] CS-1-24: total length = 2*end + interior
  [PASS] CS-1-24: two pier stations
  [PASS] CS-1-24: E-bars present (span 24 >= 22)


## CPA-1-08 — Capped Pile Abutment (Slab Bridges)

`odot.capped_pile_abutment`: SB-1-24's companion. A detailing template
(overall dimensions project-supplied) plus the TYPE-1..TYPE-5 rebar bend
legend (TYPE-6/D801 cross-references `approach_slab`'s own D801 bar).

In [12]:
from civilpy.structural.odot.capped_pile_abutment import (
    AbutmentInput, layout_capped_pile_abutment, rebar_mark,
)

d801 = rebar_mark("D801")[0]
check("CPA-1-08: D801 cross-references approach_slab", "approach_slab" in d801.note)

layout = layout_capped_pile_abutment(AbutmentInput(
    wingwall_length_ft=10.0, skew_deg=15.0, n_piles=6, pile_spacing_ft=4.0,
    footing_depth_ft=3.0,
))
check("CPA-1-08: 6 pile points generated", len(layout.pile_points) == 6)
check("CPA-1-08: wingwall springs from cap end", layout.wingwall_outline[0] == layout.cap_outline[2])
report["CPA-1-08"] = len(layout.pile_points)


  [PASS] CPA-1-08: D801 cross-references approach_slab
  [PASS] CPA-1-08: 6 pile points generated
  [PASS] CPA-1-08: wingwall springs from cap end


## CPP-1-08 — Capped Pile Pier (Continuous Slab Bridges)

`odot.capped_pile_pier`: CS-1-24's companion. Genuinely parametric (unlike
CPA-1-08/BCHW) — the sheet's own pier-length formula, fixed cap width/
end-radius; only pile count/spacing stay project-supplied.

In [13]:
from civilpy.structural.odot.capped_pile_pier import (
    PierInput, layout_capped_pile_pier, pier_length_ft,
)

L = pier_length_ft(30.0, 15.0)
check("CPP-1-08: skewed pier is longer than square (secant term)", L > pier_length_ft(30.0, 0.0))

layout = layout_capped_pile_pier(PierInput(
    slab_width_ft=30.0, skew_deg=15.0, n_piles=6, pile_spacing_ft=5.0,
))
check("CPP-1-08: cap outline is a closed stadium shape", len(layout.cap_outline) > 4)
check("CPP-1-08: 6 pile points generated", len(layout.pile_points) == 6)
report["CPP-1-08"] = len(layout.pile_points)


  [PASS] CPP-1-08: skewed pier is longer than square (secant term)
  [PASS] CPP-1-08: cap outline is a closed stadium shape
  [PASS] CPP-1-08: 6 pile points generated


## RB-1-55 — Rockers and Bolsters

`odot.rocker_bolster`: F/R capacity table (75-300 kips) plus
`layout_rocker_bolster` — tapered body, flat top (bolster) or curved-top
(rocker, TOP BEARING DETAIL radius formula).

In [15]:
from civilpy.structural.odot.rocker_bolster import (
    layout_rocker_bolster, rocker_bolster, top_bearing_plate_radius_in,
)

rb = rocker_bolster(150)
layout = layout_rocker_bolster(rb)
check("RB-1-55: bolster top narrower than base",
      (layout.bolster_top[1][0]-layout.bolster_top[0][0]) < (layout.base_outline[1][0]-layout.base_outline[0][0]))
check("RB-1-55: rocker radius derived from A",
      approx(layout.rocker_top_radius_in, top_bearing_plate_radius_in(rb.dims["A"])))

r75 = rocker_bolster(75)
check("RB-1-55: R-75 has no matching bolster", r75.bolster_no == "")
report["RB-1-55"] = 1


  [PASS] RB-1-55: bolster top narrower than base
  [PASS] RB-1-55: rocker radius derived from A
  [PASS] RB-1-55: R-75 has no matching bolster


## FB-1-82 — Fixed Bearings for Steel Beam and Girder Bridges

`odot.fixed_bearing`: F-50..F-400 pin-bearing table; `layout_fixed_bearing`
builds a self-consistent stack (base plate -> pin -> top plate).

In [16]:
from civilpy.structural.odot.fixed_bearing import fixed_bearing, layout_fixed_bearing

fb = fixed_bearing("F-150")
layout = layout_fixed_bearing(fb)
pin_bottom = layout.pin_center[2] - layout.pin_diameter_in / 2.0
check("FB-1-82: pin sits on top of the base plate (no overlap)", pin_bottom >= layout.base_thickness_in - 1e-9)
check("FB-1-82: top plate above the pin", layout.top_z_in > layout.pin_center[2])

f400 = fixed_bearing("F-400")
check("FB-1-82: F-400 requires bearing stiffeners", f400.stiffeners_required)
report["FB-1-82"] = 1


  [PASS] FB-1-82: pin sits on top of the base plate (no overlap)
  [PASS] FB-1-82: top plate above the pin
  [PASS] FB-1-82: F-400 requires bearing stiffeners


## BD-1-11 — Bearing Details for Box Beam Bridges

`odot.box_beam`'s `BeveledLoadPlate`/`load_plate_bevel` (pre-existing) plus
the new `layout_load_plate`, sized to a B1/B2 bearing pad and tilted to
the roadway grade/skew.

In [17]:
from civilpy.structural.odot.box_beam import bearing_pad, layout_load_plate

pad = bearing_pad("B1")
layout = layout_load_plate("B1", longitudinal_grade=0.04, skew_deg=20.0)
length = layout.bottom_face[1][0] - layout.bottom_face[0][0]
check("BD-1-11: plate sized to the B1 pad footprint", approx(length, pad.length))
zs = [p[2] for p in layout.top_face]
check("BD-1-11: plate top tilts with grade+skew", len(set(round(z, 6) for z in zs)) > 1)
report["BD-1-11"] = 1


  [PASS] BD-1-11: plate sized to the B1 pad footprint
  [PASS] BD-1-11: plate top tilts with grade+skew


## EXJ-4-87 / EXJ-5-93 — Strip Seal Expansion Joints

Detailing templates for a manufacturer-generic strip-seal gland:
EXJ-4-87 (steel stringers) tabulates the a1-a4 support-angle formulas;
EXJ-5-93 (box beams) tabulates the plate "A"/"B"/"C" spacing + joint-
length formula.

In [18]:
from civilpy.structural.odot.strip_seal_joint import (
    StripSealJointInput, layout_strip_seal_joint,
)
from civilpy.structural.odot.strip_seal_joint_box_beam import (
    BoxBeamJointInput, layout_box_beam_joint,
)

steel = layout_strip_seal_joint(StripSealJointInput(
    width_ft=30.0, skew_deg=20.0, stringer_stations_ft=(0.0, 7.0, 14.0, 21.0, 28.0),
    top_flange_width_in=12.0,
))
check("EXJ-4-87: one support-angle run per stringer", len(steel.support_angles) == 5)

box = layout_box_beam_joint(BoxBeamJointInput(n_beams=5, beam_width_in=48.0, skew_deg=20.0))
check("EXJ-5-93: 4 beam-gap stations for 5 beams", len(box.beam_gap_stations_ft) == 4)
report["EXJ-4-87/EXJ-5-93"] = len(steel.support_angles) + len(box.beam_gap_stations_ft)


  [PASS] EXJ-4-87: one support-angle run per stringer
  [PASS] EXJ-5-93: 4 beam-gap stations for 5 beams


## Rhino layer taxonomy — `civilpy.structural.rhino_layers`

Confirms the shared layer-path constants match `Core/Gdr.cs`
(RhinoODOTExtension, commit `34f7051`) exactly, and that `ensure_layer`
builds the same nested tree in an offline `rhino3dm.File3dm` that the C#
plugin builds in a live document.

In [19]:
import rhino3dm
from civilpy.structural import rhino_layers as rl

f = rhino3dm.File3dm()
idx = rl.ensure_layer(f, rl.LAYER_BOX_BEAMS)
check("rhino_layers: Superstructure::Box Beams created", f.Layers[idx].FullPath == "Superstructure::Box Beams")
idx2 = rl.ensure_layer(f, rl.LAYER_GIRDERS)
parents = {l.FullPath for l in f.Layers if l.FullPath == "Superstructure"}
check("rhino_layers: Superstructure parent shared across leaves", len(parents) == 1)
report["rhino_layers"] = len(list(f.Layers))


  [PASS] rhino_layers: Superstructure::Box Beams created


  [PASS] rhino_layers: Superstructure parent shared across leaves


## Summary

In [20]:
print("All SCD components exercised successfully:")
for k, v in report.items():
    print(f"  {k:24s}: {v}")
print(f"\n{len(report)} components checked, 0 failures (any assertion failure would have stopped this notebook above).")


All SCD components exercised successfully:
  AS-1-15                 : 266
  DS-1-92                 : 7
  PCB-91                  : 4
  AS-2-15                 : 33
  HW-2.1/HW-2.2           : 2
  HW-1.1                  : 2
  BCHW                    : 3
  bridge_railing          : 14
  SB-1-24                 : 159
  CS-1-24                 : 169
  CPA-1-08                : 6
  CPP-1-08                : 6
  A-1-20                  : 1
  RB-1-55                 : 1
  FB-1-82                 : 1
  BD-1-11                 : 1
  EXJ-4-87/EXJ-5-93       : 9
  rhino_layers            : 3

18 components checked, 0 failures (any assertion failure would have stopped this notebook above).
